In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("../data/creditcard.csv")

In [3]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# DIVISÃO TREINO/TESTE
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [6]:
# =========================
# INSTALAR PYCARET
# =========================
#%pip install pycaret

# =========================
# IMPORTS
# =========================
import pandas as pd

from sklearn.model_selection import train_test_split
from pycaret.classification import setup, compare_models, predict_model, pull
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

# =========================
# PYCARET
# =========================

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

exp = setup(
    data=train_df,
    test_data=test_df,
    target="Class",
    index=False,  # PyCaret assigns a fresh index
    session_id=42,
    verbose=False
)


# Retorna os 10 melhores modelos treinados
modelos_treinados = compare_models(n_select=10)

# =========================
# LEADERBOARD INTERNO DO PYCARET
# =========================
print("\n===== LEADERBOARD (Validação PyCaret) =====")
print(pull())

# =========================
# MÉTRICAS DE TODOS OS MODELOS (No Test Set)
# =========================
resultados = []

for model in modelos_treinados:
    model_name = model.__class__.__name__

    # Previsão no conjunto de teste
    preds_df = predict_model(model, data=test_df, verbose=False)

    y_true = test_df['Class'].values
    # No PyCaret 3.x, a coluna com o rótulo previsto é 'prediction_label'
    y_pred = preds_df['prediction_label'].astype(int).values

    # Métricas
    acc = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
    f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

    resultados.append({
        "Modelo": model_name,
        "Tipo": model_name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score Weighted": f1_weighted,
        "F1-score": f1
    })

# =========================
# RESULTADOS
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="F1-score Weighted", ascending=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,0.9996,0.9831,0.7915,0.9506,0.8617,0.8615,0.8662,0.4000
rf,Random Forest Classifier,0.9995,0.9501,0.7738,0.9470,0.8493,0.8491,0.8546,17.5850
et,Extra Trees Classifier,0.9995,0.9525,0.7687,0.9578,0.8508,0.8505,0.8568,1.6430
lda,Linear Discriminant Analysis,0.9994,0.9808,0.7587,0.8788,0.8129,0.8126,0.8155,0.0900
lr,Logistic Regression,0.9992,0.9633,0.6396,0.8646,0.7340,0.7336,0.7426,3.5170
ada,Ada Boost Classifier,0.9992,0.9724,0.7074,0.7947,0.7460,0.7456,0.7481,5.8060
dt,Decision Tree Classifier,0.9991,0.8867,0.7738,0.7477,0.7575,0.7571,0.7587,1.9150
ridge,Ridge Classifier,0.9989,0.9808,0.4138,0.8591,0.5574,0.5569,0.5951,0.0370
gbc,Gradient Boosting Classifier,0.9988,0.6010,0.4328,0.7367,0.5153,0.5148,0.5463,30.3250
knn,K Neighbors Classifier,0.9983,0.5887,0.0482,0.8750,0.0902,0.0900,0.1985,4.8600



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
xgboost         Extreme Gradient Boosting    0.9996  0.9831  0.7915  0.9506   
rf               Random Forest Classifier    0.9995  0.9501  0.7738  0.9470   
et                 Extra Trees Classifier    0.9995  0.9525  0.7687  0.9578   
lda          Linear Discriminant Analysis    0.9994  0.9808  0.7587  0.8788   
lr                    Logistic Regression    0.9992  0.9633  0.6396  0.8646   
ada                  Ada Boost Classifier    0.9992  0.9724  0.7074  0.7947   
dt               Decision Tree Classifier    0.9991  0.8867  0.7738  0.7477   
ridge                    Ridge Classifier    0.9989  0.9808  0.4138  0.8591   
gbc          Gradient Boosting Classifier    0.9988  0.6010  0.4328  0.7367   
knn                K Neighbors Classifier    0.9983  0.5887  0.0482  0.8750   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
svm    

In [7]:
# =========================
# INSTALAR H2O
# =========================
%pip install h2o

# =========================
# IMPORTS
# =========================
import pandas as pd
import h2o
from h2o.automl import H2OAutoML

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# INICIAR H2O
# =========================
h2o.init()

# =========================
# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

# =========================
# PARA H2O
# =========================
train_h2o = h2o.H2OFrame(train_df)
test_h2o = h2o.H2OFrame(test_df)

train_h2o['Class'] = train_h2o['Class'].asfactor()
test_h2o['Class'] = test_h2o['Class'].asfactor()

x = train_h2o.columns[:-1]
y = 'Class'

# =========================
# AUTO ML
# =========================
aml = H2OAutoML(
    max_models=20,
    seed=42,
    balance_classes=True
)

aml.train(x=x, y=y, training_frame=train_h2o)

# =========================
# LEADERBOARD
# =========================
print("\n===== LEADERBOARD =====")
print(aml.leaderboard)

# =========================
# MÉTRICAS DE TODOS OS MODELOS
# =========================
leaderboard = aml.leaderboard.as_data_frame()

resultados = []

for model_id in leaderboard['model_id']:

    model = h2o.get_model(model_id)

    # previsão
    preds = model.predict(test_h2o).as_data_frame()

    y_true = test_df['Class'].values
    y_pred = preds['predict'].astype(int).values

    # métricas
    acc = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
    f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

    resultados.append({
        "Modelo": model_id,
        "Tipo": model.algo,
        "Accuracy": round(acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-score": round(f1, 4),
        "F1-score Weighted": round(f1_weighted, 4)
    })

# =========================
# RESULTADO FINAL
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="F1-score Weighted", ascending=False))

Note: you may need to restart the kernel to use updated packages.
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.20" 2026-07-21; OpenJDK Runtime Environment Homebrew (build 17.0.20+0); OpenJDK 64-Bit Server VM Homebrew (build 17.0.20+0, mixed mode, sharing)
  Starting server from /Users/lannaces/Documents/TCC/.venv311/lib/python3.11/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/x_/pw0p233x3n5_ny9bsbpb0ldc0000gn/T/tmp82ij5ffi
  JVM stdout: /var/folders/x_/pw0p233x3n5_ny9bsbpb0ldc0000gn/T/tmp82ij5ffi/h2o_lannaces_started_from_python.out
  JVM stderr: /var/folders/x_/pw0p233x3n5_ny9bsbpb0ldc0000gn/T/tmp82ij5ffi/h2o_lannaces_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 24 days
H2O_cluster_name:,H2O_from_python_lannaces_gfr9u3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
22:09:43.389: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%

===== LEADERBOARD =====
model

In [ ]:
# =========================
# INSTALAR LAZYPREDICT
# =========================
%pip install lazypredict

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lazypredict.Supervised import LazyClassifier

# =========================
# LAZYPREDICT
# =========================
clf = LazyClassifier(verbose=0, ignore_warnings=True)

models, predictions = clf.fit(X_train, X_test, y_train, y_test)

# =========================
# RESULTADOS
# =========================
print("\n===== RESULTADOS =====")
print(models)

print("\n===== MÉTRICAS =====")
all_metrics = []

for model_name, model in clf.models.items():
    y_pred = model.predict(X_test)

    all_metrics.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1 Score Weighted": f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, pos_label=1, zero_division=0)

    })

metrics_df = (
    pd.DataFrame(all_metrics)
    .sort_values("F1 Score Weighted", ascending=False)
    .reset_index(drop=True)
)

print(metrics_df)

Note: you may need to restart the kernel to use updated packages.


Large dataset detected (227845 samples). Training all models may take a long time. Consider using a subset or setting max_models/timeout.


In [4]:
# =========================
# INSTALAR FLAML
# =========================
%pip install "flaml[automl]"

# =========================
# IMPORTS
# =========================
from flaml import AutoML

print("Dataset carregado:")
print(df.head())

# =========================
# TREINAMENTO COM FLAML
# =========================
automl = AutoML()

settings = {
    "time_budget": 60,
    "metric": 'accuracy',
    "task": 'classification',
    "log_file_name": "flaml.log",
}

automl.fit(X_train=X_train, y_train=y_train, **settings)

# =========================
# RESULTADOS
# =========================
print("\n===== RESULTADOS =====")
print("Melhor modelo:", automl.best_estimator)
print("Melhores parâmetros:", automl.best_config)
print("Melhor loss:", automl.best_loss)

# =========================
# AVALIAÇÃO
# =========================
y_pred = automl.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score Weighted:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4))

Note: you may need to restart the kernel to use updated packages.
Dataset carregado:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -